In [2]:
import os
import torch
from pymilvus import connections, Collection
from transformers import AutoModel, AutoTokenizer
import ollama

# ========================
# CONFIGURAÇÃO DO MILVUS
# ========================
connections.connect("default", host="127.0.0.1", port="19530")
COLLECTION_NAME = "rag_embeddings_milvus"
collection = Collection(COLLECTION_NAME)

# ========================
# EMBEDDINGS (mesmo modelo usado na ingestão)
# ========================
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def get_embedding(text: str):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
        embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings[0].numpy().tolist()

# ========================
# FUNÇÃO DE RECUPERAÇÃO DO CONTEXTO
# ========================
def retrieve_context(query: str, top_k: int = 5):
    query_emb = get_embedding(query)

    collection.load()
    results = collection.search(
        data=[query_emb],
        anns_field="embedding",
        param={"metric_type": "IP", "params": {"nprobe": 10}},
        limit=top_k,
        output_fields=["source_file", "source_url", "chunk_index", "chunk_text"]
    )

    contexts = []
    refs = []
    for r in results[0]:
        chunk_text = r.entity.get("chunk_text")
        source = r.entity.get("source_file")
        url = r.entity.get("source_url")
        contexts.append(chunk_text)
        refs.append(f"📄 {source} | 🔗 {url}")

    return "\n\n".join(contexts), "\n".join(refs)

# ========================
# FUNÇÃO DE GERAÇÃO DE RESPOSTA (via Ollama + Mistral)
# ========================
def generate_answer(query: str, context: str):
    prompt = f"""
Você é um assistente técnico especializado em licenciamento ambiental (EIA/RIMA).
Responda à pergunta do usuário **usando apenas o contexto fornecido**.

Contexto:
{context}

Pergunta:
{query}

Responda de forma clara, objetiva e técnica.
"""
    response = ollama.chat(
        model="mistral:7b",
        messages=[
            {"role": "system", "content": "Você é um assistente técnico ambiental especializado em EIA/RIMA."},
            {"role": "user", "content": prompt}
        ]
    )
    return response["message"]["content"]

# ========================
# LOOP DE CHAT
# ========================
print("🤖 Chatbot EIA/RIMA usando Milvus + Mistral 7B (Ollama). Digite 'sair' para encerrar.\n")

while True:
    user_input = input("Você: ")
    if user_input.lower() in ["sair", "exit", "quit"]:
        break

    # Recupera contexto
    context, refs = retrieve_context(user_input)

    # Gera resposta
    answer = generate_answer(user_input, context)

    print("\nBot:", answer)
    print("\n--- Fontes ---")
    print(refs)
    print("\n")


🤖 Chatbot EIA/RIMA usando Milvus + Mistral 7B (Ollama). Digite 'sair' para encerrar.



Você:  Estamos no processo de duplicação da SP-97 e necessitamos retirar a vegetação local na extensão da estrada. Como devemos proceder de acordo com a legislação local?



Bot:  Para retirar a vegetação local durante o processo de duplicação da SP-97, você precisará seguir alguns passos para se adequar à legislação local:

1. A primeira etapa é avaliar a relevância e importância ecológica da área em questão. Considerando que se trata de uma zona próxima ao Bioma Mata Atlântica, é importante evitar a destruição excessiva de vegetação nativa.

2. Conforme o texto fornecido, o artigo 4o., inciso III, da Lei no 6.938, de 31 de agosto de 1981; e o artigo 2o da Lei no 10.650, de 16 de abril de 2003 alteram as dispositivos aos Decretos nos 3.179, de 21 de setembro de 1999, e 3.420, de 20 de abril de 2000, que dispõe sobre a proteção da vegetação nativa. É importante seguir as normas estabelecidas nestes documentos.

3. Neste caso específico, onde se trata de retirar vegetação local para construção de infraestruturas, é necessário realizar um estudo ambiental (EIA) para avaliar os impactos potenciais no meio ambiente e suas possíveis medidas de mitigação.

4. A

Você:  sait



Bot:  Para entender a situação apresentada neste documento, é necessário realizar um análise detalhada dos elementos envolvidos, como o local do empreendimento, as características físicas e legais das terras em questão, bem como os processos de aprovação e avaliação ambiental.

O empreendimento em questão se encontra localizado no Estrada Municipal Roque Antunes Fragoso, na cidade de Boituva, Estado de São Paulo, Brasil. A área abrangida pelo projeto possui as seguintes coordenadas geográficas: -23.2701883/-47.6286933 à -23.28155/-47.621855.

A empresa responsável pelo projeto é a S.A - URBANISMO & ARQUITETURA – EIRELI, que apresentou uma procuração para o local em questão. A área do empreendimento tem uma superfície total de aproximadamente 43298,57 m².

O documento apresenta também a localização dos lotes envolvidos no projeto, bem como os relatórios de avaliação de impacto ambiental e arqueológico associados a cada lote. Além disso, o documento apresenta as legenda urbanística (lim

Você:  sair
